# TCGA-BRCA Cohort Blueprint Review

This notebook is review-only. It loads the latest saved cohort blueprint outputs from disk, checks the run-level validation state, and writes review tables for human source audit and study-design review.


## Load the latest saved cohort blueprint run


In [1]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / '.git').exists():
            return candidate
    raise FileNotFoundError('Could not locate the repository root from the current working directory.')


repo_root = find_repo_root(Path.cwd())
latest_pointer_path = (
    repo_root
    / '01-data'
    / 'audit'
    / 'tcga-brca'
    / 'cohort'
    / 'tcga_brca_cohort_blueprint_latest.json'
)
if not latest_pointer_path.exists():
    raise FileNotFoundError(
        f'Latest cohort blueprint pointer not found: {latest_pointer_path}. Run the cohort blueprint script first.'
    )

latest_pointer = json.loads(latest_pointer_path.read_text(encoding='utf-8'))
required_path = repo_root / latest_pointer['cohort_blueprint_required_fields_tsv']
optional_path = repo_root / latest_pointer['cohort_blueprint_optional_fields_tsv']
deferred_path = repo_root / latest_pointer['cohort_blueprint_deferred_fields_tsv']
join_path_path = repo_root / latest_pointer['cohort_blueprint_join_path_tsv']
ambiguities_path = repo_root / latest_pointer['cohort_blueprint_ambiguities_tsv']
summary_path = repo_root / latest_pointer['cohort_blueprint_summary_tsv']
run_log_path = repo_root / latest_pointer['run_log_json']
results_root = repo_root / '09-trials' / '01-tcga-only-source-audited' / '05-results'
results_root.mkdir(parents=True, exist_ok=True)

display(pd.DataFrame([latest_pointer]))


,updated_at_utc,blueprint_run_id,shortlist_run_id,core_audit_run_id,clinical_parse_run_id,endpoint_crosswalk_run_id,biospecimen_crosswalk_run_id,biospecimen_parse_run_id,clinical_source_run_id,biospecimen_source_run_id,...,cohort_blueprint_required_fields_tsv,cohort_blueprint_optional_fields_tsv,cohort_blueprint_deferred_fields_tsv,cohort_blueprint_join_path_tsv,cohort_blueprint_ambiguities_tsv,cohort_blueprint_summary_tsv,run_log_json,clinical_shortlist_latest_json,endpoint_crosswalk_latest_json,biospecimen_crosswalk_latest_json
0,2026-04-13T08:18:35Z,20260413T081835Z,20260412T033602Z,20260412T023839Z,20260412T010932Z,20260412T042036Z,20260413T073640Z,20260413T070359Z,20260412T000556Z,20260412T000556Z,...,01-data/audit/tcga-brca/cohort/cohort_blueprin...,01-data/audit/tcga-brca/cohort/cohort_blueprin...,01-data/audit/tcga-brca/cohort/cohort_blueprin...,01-data/audit/tcga-brca/cohort/cohort_blueprin...,01-data/audit/tcga-brca/cohort/cohort_blueprin...,01-data/audit/tcga-brca/cohort/cohort_blueprin...,01-data/audit/tcga-brca/cohort/cohort_blueprin...,01-data/audit/tcga-brca/variables/tcga_brca_cl...,01-data/audit/tcga-brca/variables/tcga_brca_en...,01-data/audit/tcga-brca/variables/tcga_brca_bi...


## Load saved cohort blueprint artifacts


In [2]:
required_df = pd.read_csv(required_path, sep='\t')
optional_df = pd.read_csv(optional_path, sep='\t')
deferred_df = pd.read_csv(deferred_path, sep='\t')
join_path_df = pd.read_csv(join_path_path, sep='\t')
ambiguity_df = pd.read_csv(ambiguities_path, sep='\t')
summary_df = pd.read_csv(summary_path, sep='\t')
run_log = json.loads(run_log_path.read_text(encoding='utf-8'))

field_bucket_order = [
    'required_for_first_baseline',
    'optional_for_first_baseline',
    'deferred_for_later_expansion',
]
source_layer_order = ['clinical', 'endpoint', 'biospecimen']
join_strength_order = ['strong_current_evidence', 'workable_but_needs_review', 'deferred']
link_timing_order = ['link_now', 'link_later', 'prepare_only']
severity_order = ['high', 'medium', 'low']

for frame in [required_df, optional_df, deferred_df]:
    frame['field_bucket'] = pd.Categorical(frame['field_bucket'], categories=field_bucket_order, ordered=True)
    frame['source_layer'] = pd.Categorical(frame['source_layer'], categories=source_layer_order, ordered=True)

join_path_df['join_strength'] = pd.Categorical(
    join_path_df['join_strength'], categories=join_strength_order, ordered=True
)
join_path_df['link_timing'] = pd.Categorical(
    join_path_df['link_timing'], categories=link_timing_order, ordered=True
)
ambiguity_df['severity'] = pd.Categorical(
    ambiguity_df['severity'], categories=severity_order, ordered=True
)

unit_rows = summary_df.loc[
    summary_df['summary_metric'].isin(['proposed_unit_of_analysis', 'proposed_biospecimen_anchor'])
].reset_index(drop=True)
strong_join_df = join_path_df.loc[
    join_path_df['join_strength'] == 'strong_current_evidence'
].reset_index(drop=True)
blocking_ambiguity_df = ambiguity_df.loc[
    ambiguity_df['blocks_final_cohort_construction'] == True
].sort_values(['severity', 'ambiguity_id']).reset_index(drop=True)

print(f'Required fields TSV: {required_path}')
print(f'Optional fields TSV: {optional_path}')
print(f'Deferred fields TSV: {deferred_path}')
print(f'Join path TSV: {join_path_path}')
print(f'Ambiguities TSV: {ambiguities_path}')
print(f'Summary TSV: {summary_path}')
print(f'Run log: {run_log_path}')
print(f"Blueprint run ID: {latest_pointer['blueprint_run_id']}")
display(pd.DataFrame([run_log['validation']]))
display(unit_rows)
display(strong_join_df)
display(blocking_ambiguity_df)


Required fields TSV: D:\Projects\brcapath-rx\01-data\audit\tcga-brca\cohort\cohort_blueprint_runs\20260413T081835Z\cohort_blueprint_required_fields.tsv
Optional fields TSV: D:\Projects\brcapath-rx\01-data\audit\tcga-brca\cohort\cohort_blueprint_runs\20260413T081835Z\cohort_blueprint_optional_fields.tsv
Deferred fields TSV: D:\Projects\brcapath-rx\01-data\audit\tcga-brca\cohort\cohort_blueprint_runs\20260413T081835Z\cohort_blueprint_deferred_fields.tsv
Join path TSV: D:\Projects\brcapath-rx\01-data\audit\tcga-brca\cohort\cohort_blueprint_runs\20260413T081835Z\cohort_blueprint_join_path.tsv
Ambiguities TSV: D:\Projects\brcapath-rx\01-data\audit\tcga-brca\cohort\cohort_blueprint_runs\20260413T081835Z\cohort_blueprint_ambiguities.tsv
Summary TSV: D:\Projects\brcapath-rx\01-data\audit\tcga-brca\cohort\cohort_blueprint_runs\20260413T081835Z\cohort_blueprint_summary.tsv
Run log: D:\Projects\brcapath-rx\01-data\audit\tcga-brca\cohort\cohort_blueprint_runs\20260413T081835Z\run_log.json
Blueprin

,passed,clinical_shortlist_latest_pointer_found,endpoint_crosswalk_latest_pointer_found,biospecimen_crosswalk_latest_pointer_found,clinical_shortlist_run_log_completed,clinical_core_audit_run_log_completed,clinical_biotab_run_log_completed,source_run_log_completed,endpoint_crosswalk_run_log_completed,biospecimen_crosswalk_run_log_completed,...,optional_fields_row_count_positive,deferred_fields_row_count_positive,join_path_row_count_positive,ambiguities_row_count_positive,summary_row_count_positive,field_assignment_disjoint,join_strength_values_valid,ambiguity_severity_values_valid,no_prior_run_overwrite,latest_pointer_written_after_success_only
0,True,True,True,True,True,True,True,True,True,True,...,True,True,True,True,True,True,True,True,True,True


,blueprint_run_id,summary_section,summary_metric,summary_value,notes
0,20260413T081835Z,design,proposed_unit_of_analysis,patient/case,The first baseline blueprint keeps the analysi...
1,20260413T081835Z,design,proposed_biospecimen_anchor,sample,biospecimen_sample is the required specimen an...


,blueprint_run_id,step_order,left_node,right_node,left_table_name,right_table_name,left_field_name,right_field_name,join_keys_or_evidence,join_strength,link_timing,rationale,ambiguity_flag,notes_placeholder
0,20260413T081835Z,1,patient/case,clinical_patient,NaN,clinical_patient,case_identifier,bcr_patient_barcode / bcr_patient_uuid,clinical_patient carries both audited case ide...,strong_current_evidence,link_now,Use clinical_patient as the central case-level...,False,[fill in during cohort blueprint review]
1,20260413T081835Z,4,patient/case,clinical_patient endpoint-like fields,clinical_patient,clinical_patient,bcr_patient_barcode / bcr_patient_uuid,vital_status / last_contact_days_to / tumor_st...,clinical_patient endpoint-like fields already ...,strong_current_evidence,prepare_only,Patient-table endpoint candidates can be carri...,False,[fill in during cohort blueprint review]
2,20260413T081835Z,7,sample,portion,biospecimen_sample,biospecimen_portion,bcr_sample_barcode / bcr_sample_uuid,bcr_sample_barcode / bcr_portion_barcode / bcr...,biospecimen_portion repeats bcr_sample_barcode...,strong_current_evidence,link_later,Sample-to-portion is the strongest saved child...,False,[fill in during cohort blueprint review]


,blueprint_run_id,ambiguity_id,ambiguity_category,source_layer,table_name,field_name_or_step,severity,why_unresolved,current_saved_evidence,proposed_blueprint_handling,blocks_final_cohort_construction,notes_placeholder
0,20260413T081835Z,A01,endpoint_overlap_patient_vs_followup,endpoint,clinical_patient | clinical_follow_up_v4_0,vital_status,high,The same endpoint-like field exists in both pa...,clinical_patient.vital_status has 1097 non-mis...,Carry both rows as endpoint join-preparation f...,True,[fill in during cohort blueprint review]
1,20260413T081835Z,A02,endpoint_overlap_patient_vs_followup,endpoint,clinical_patient | clinical_follow_up_v4_0,last_contact_days_to,high,Patient and follow-up tables both carry last-c...,clinical_patient.last_contact_days_to has 993 ...,Keep both rows optional and endpoint-preparato...,True,[fill in during cohort blueprint review]
2,20260413T081835Z,A03,endpoint_overlap_patient_vs_followup,endpoint,clinical_patient | clinical_follow_up_v4_0,tumor_status,high,Tumor-status signals overlap across patient an...,clinical_patient.tumor_status has 972 non-miss...,Preserve both rows for later endpoint review a...,True,[fill in during cohort blueprint review]
3,20260413T081835Z,A04,endpoint_overlap_patient_vs_followup,endpoint,clinical_patient | clinical_follow_up_v4_0,new_tumor_event_dx_indicator,high,New-tumor-event indicators overlap across pati...,clinical_patient.new_tumor_event_dx_indicator ...,Keep both rows as later endpoint-reconciliatio...,True,[fill in during cohort blueprint review]
4,20260413T081835Z,A05,sparse_endpoint_timing,endpoint,clinical_patient | clinical_follow_up_v4_0,death_days_to,high,Death timing exists in both tables but remains...,clinical_patient.death_days_to has 104 non-mis...,Record death timing as deferred endpoint evide...,True,[fill in during cohort blueprint review]
5,20260413T081835Z,A06,all_missing_progression_timing,endpoint,clinical_patient,days_to_patient_progression_free | days_to_tum...,high,Progression timing fields were surfaced only a...,Both progression timing fields have 0 non-miss...,Keep both fields deferred and exclude them fro...,True,[fill in during cohort blueprint review]
6,20260413T081835Z,A07,clinical_vs_biospecimen_patient_identifiers,clinical | biospecimen,clinical_patient | biospecimen_sample | biospe...,bcr_patient_barcode | bcr_patient_uuid,high,Clinical and biospecimen layers expose both pa...,clinical_patient has 1097 non-missing values f...,Use both identifier forms as evidence-bearing ...,True,[fill in during cohort blueprint review]
7,20260413T081835Z,A08,case_count_mismatch,clinical | source | biospecimen,clinical_patient | source_metadata | biospecim...,case coverage counts,high,"The saved clinical, source-acquisition, and bi...",clinical_patient row count is 1097; clinical s...,Flag cross-layer case counts as a blocking rec...,True,[fill in during cohort blueprint review]


## Save required blueprint review table


In [3]:
field_review_columns = [
    'field_bucket',
    'source_layer',
    'table_name',
    'field_name',
    'proposed_role',
    'rationale',
    'dependency_for_join',
    'ambiguity_flag',
    'upstream_source_type',
    'upstream_bucket_or_role',
    'upstream_rule',
    'manual_review_priority',
    'missing_like_fraction',
    'non_missing_count',
    'distinct_non_missing_count',
    'paired_field_name',
    'evidence_rule',
    'evidence_rate',
    'notes_placeholder',
]

required_review_df = (
    required_df.loc[:, field_review_columns]
    .sort_values(['source_layer', 'table_name', 'field_name'], ascending=[True, True, True])
    .reset_index(drop=True)
)
required_review_path = results_root / '48_cohort_blueprint_required_fields.tsv'
required_review_df.to_csv(required_review_path, sep='\t', index=False)

print(f'Saved: {required_review_path}')
display(required_review_df)


Saved: D:\Projects\brcapath-rx\09-trials\01-tcga-only-source-audited\05-results\48_cohort_blueprint_required_fields.tsv


,field_bucket,source_layer,table_name,field_name,proposed_role,rationale,dependency_for_join,ambiguity_flag,upstream_source_type,upstream_bucket_or_role,upstream_rule,manual_review_priority,missing_like_fraction,non_missing_count,distinct_non_missing_count,paired_field_name,evidence_rule,evidence_rate,notes_placeholder
0,required_for_first_baseline,clinical,clinical_patient,age_at_diagnosis,baseline_covariate,Carry forward audited usable baseline fields a...,case_level_baseline_core,False,clinical_shortlist,usable_baseline,baseline_candidate_structured_core_field,low,0.000000,1097,65,NaN,NaN,NaN,[fill in during cohort blueprint review]
1,required_for_first_baseline,clinical,clinical_patient,ajcc_metastasis_pathologic_pm,baseline_covariate,Carry forward audited usable baseline fields a...,case_level_baseline_core,False,clinical_shortlist,usable_baseline,baseline_candidate_structured_core_field,low,0.000000,1097,4,NaN,NaN,NaN,[fill in during cohort blueprint review]
2,required_for_first_baseline,clinical,clinical_patient,ajcc_nodes_pathologic_pn,baseline_covariate,Carry forward audited usable baseline fields a...,case_level_baseline_core,False,clinical_shortlist,usable_baseline,baseline_candidate_structured_core_field,low,0.000000,1097,16,NaN,NaN,NaN,[fill in during cohort blueprint review]
3,required_for_first_baseline,clinical,clinical_patient,ajcc_pathologic_tumor_stage,baseline_covariate,Carry forward audited usable baseline fields a...,case_level_baseline_core,False,clinical_shortlist,usable_baseline,baseline_candidate_structured_core_field,low,0.010027,1086,12,NaN,NaN,NaN,[fill in during cohort blueprint review]
4,required_for_first_baseline,clinical,clinical_patient,ajcc_staging_edition,baseline_covariate,Carry forward audited usable baseline fields a...,case_level_baseline_core,False,clinical_shortlist,usable_baseline,baseline_candidate_structured_core_field,medium,0.128532,956,5,NaN,NaN,NaN,[fill in during cohort blueprint review]
5,required_for_first_baseline,clinical,clinical_patient,ajcc_tumor_pathologic_pt,baseline_covariate,Carry forward audited usable baseline fields a...,case_level_baseline_core,False,clinical_shortlist,usable_baseline,baseline_candidate_structured_core_field,low,0.000000,1097,13,NaN,NaN,NaN,[fill in during cohort blueprint review]
6,required_for_first_baseline,clinical,clinical_patient,anatomic_neoplasm_subdivision,baseline_covariate,Carry forward audited usable baseline fields a...,case_level_baseline_core,False,clinical_shortlist,usable_baseline,baseline_candidate_structured_core_field,low,0.000000,1097,53,NaN,NaN,NaN,[fill in during cohort blueprint review]
7,required_for_first_baseline,clinical,clinical_patient,axillary_staging_method,baseline_covariate,Carry forward audited usable baseline fields a...,case_level_baseline_core,False,clinical_shortlist,usable_baseline,baseline_candidate_structured_core_field,medium,0.196901,881,5,NaN,NaN,NaN,[fill in during cohort blueprint review]
8,required_for_first_baseline,clinical,clinical_patient,bcr_patient_barcode,case_join_identifier,Retain both audited clinical patient identifie...,case_to_clinical_and_case_to_followup_preparation,False,clinical_core_field_audit,manual_blueprint_backbone,cohort_blueprint_required_case_identifier,NaN,0.000000,1097,1097,NaN,NaN,NaN,[fill in during cohort blueprint review]
9,required_for_first_baseline,clinical,clinical_patient,bcr_patient_uuid,case_join_identifier,Retain both audited clinical patient identifie...,case_to_clinical_and_case_to_followup_preparation,False,clinical_core_field_audit,manual_blueprint_backbone,cohort_blueprint_required_case_identifier,NaN,0.000000,1097,1097,NaN,NaN,NaN,[fill in during cohort blueprint review]


## Save optional and deferred blueprint review tables


In [4]:
optional_review_df = (
    optional_df.loc[:, field_review_columns]
    .sort_values(['source_layer', 'table_name', 'field_name'], ascending=[True, True, True])
    .reset_index(drop=True)
)
optional_review_path = results_root / '49_cohort_blueprint_optional_fields.tsv'
optional_review_df.to_csv(optional_review_path, sep='\t', index=False)

deferred_review_df = (
    deferred_df.loc[:, field_review_columns]
    .sort_values(['source_layer', 'table_name', 'field_name'], ascending=[True, True, True])
    .reset_index(drop=True)
)
deferred_review_path = results_root / '50_cohort_blueprint_deferred_fields.tsv'
deferred_review_df.to_csv(deferred_review_path, sep='\t', index=False)

print(f'Saved: {optional_review_path}')
print(f'Saved: {deferred_review_path}')
display(optional_review_df)
display(deferred_review_df)


Saved: D:\Projects\brcapath-rx\09-trials\01-tcga-only-source-audited\05-results\49_cohort_blueprint_optional_fields.tsv
Saved: D:\Projects\brcapath-rx\09-trials\01-tcga-only-source-audited\05-results\50_cohort_blueprint_deferred_fields.tsv


,field_bucket,source_layer,table_name,field_name,proposed_role,rationale,dependency_for_join,ambiguity_flag,upstream_source_type,upstream_bucket_or_role,upstream_rule,manual_review_priority,missing_like_fraction,non_missing_count,distinct_non_missing_count,paired_field_name,evidence_rule,evidence_rate,notes_placeholder
0,optional_for_first_baseline,clinical,clinical_drug,bcr_patient_barcode,case_join_identifier,Keep the audited patient and follow-up identif...,case_to_treatment_optional_link,False,clinical_core_field_audit,manual_blueprint_join_support,cohort_blueprint_optional_join_identifier,NaN,0.000000,2406,780,NaN,NaN,NaN,[fill in during cohort blueprint review]
1,optional_for_first_baseline,clinical,clinical_drug,bcr_patient_uuid,case_join_identifier,Keep the audited patient and follow-up identif...,case_to_treatment_optional_link,False,clinical_core_field_audit,manual_blueprint_join_support,cohort_blueprint_optional_join_identifier,NaN,0.000000,2406,780,NaN,NaN,NaN,[fill in during cohort blueprint review]
2,optional_for_first_baseline,clinical,clinical_drug,pharmaceutical_therapy_drug_name,treatment_proxy_field,Preserve audited treatment-proxy fields as opt...,optional_case_level_treatment_proxy,True,clinical_shortlist,usable_treatment_proxy,treatment_proxy_direct_treatment_table,medium,0.006650,2390,203,NaN,NaN,NaN,[fill in during cohort blueprint review]
3,optional_for_first_baseline,clinical,clinical_drug,pharmaceutical_therapy_type,treatment_proxy_field,Preserve audited treatment-proxy fields as opt...,optional_case_level_treatment_proxy,True,clinical_shortlist,usable_treatment_proxy,treatment_proxy_direct_treatment_table,medium,0.002909,2399,8,NaN,NaN,NaN,[fill in during cohort blueprint review]
4,optional_for_first_baseline,clinical,clinical_drug,pharmaceutical_tx_ended_days_to,treatment_proxy_field,Preserve audited treatment-proxy fields as opt...,optional_case_level_treatment_proxy,True,clinical_shortlist,usable_treatment_proxy,treatment_proxy_direct_treatment_table,medium,0.249792,1805,504,NaN,NaN,NaN,[fill in during cohort blueprint review]
5,optional_for_first_baseline,clinical,clinical_drug,pharmaceutical_tx_ongoing_indicator,treatment_proxy_field,Preserve audited treatment-proxy fields as opt...,optional_case_level_treatment_proxy,True,clinical_shortlist,usable_treatment_proxy,treatment_proxy_direct_treatment_table,medium,0.006234,2391,2,NaN,NaN,NaN,[fill in during cohort blueprint review]
6,optional_for_first_baseline,clinical,clinical_drug,pharmaceutical_tx_started_days_to,treatment_proxy_field,Preserve audited treatment-proxy fields as opt...,optional_case_level_treatment_proxy,True,clinical_shortlist,usable_treatment_proxy,treatment_proxy_direct_treatment_table,medium,0.049044,2288,486,NaN,NaN,NaN,[fill in during cohort blueprint review]
7,optional_for_first_baseline,clinical,clinical_follow_up_v4_0,bcr_followup_barcode,followup_join_identifier,Keep the audited patient and follow-up identif...,case_to_followup_or_case_to_treatment_optional...,True,clinical_core_field_audit,manual_blueprint_join_support,cohort_blueprint_optional_join_identifier,NaN,0.000000,716,716,NaN,NaN,NaN,[fill in during cohort blueprint review]
8,optional_for_first_baseline,clinical,clinical_follow_up_v4_0,bcr_followup_uuid,followup_join_identifier,Keep the audited patient and follow-up identif...,case_to_followup_or_case_to_treatment_optional...,True,clinical_core_field_audit,manual_blueprint_join_support,cohort_blueprint_optional_join_identifier,NaN,0.000000,716,716,NaN,NaN,NaN,[fill in during cohort blueprint review]
9,optional_for_first_baseline,clinical,clinical_follow_up_v4_0,bcr_patient_barcode,followup_join_identifier,Keep the audited patient and follow-up identif...,case_to_followup_or_case_to_treatment_optional...,True,clinical_core_field_audit,manual_blueprint_join_support,cohort_blueprint_optional_join_identifier,NaN,0.000000,716,619,NaN,NaN,NaN,[fill in during cohort blueprint review]


,field_bucket,source_layer,table_name,field_name,proposed_role,rationale,dependency_for_join,ambiguity_flag,upstream_source_type,upstream_bucket_or_role,upstream_rule,manual_review_priority,missing_like_fraction,non_missing_count,distinct_non_missing_count,paired_field_name,evidence_rule,evidence_rate,notes_placeholder
0,deferred_for_later_expansion,endpoint,clinical_follow_up_v4_0,death_days_to,deferred_endpoint_candidate,Defer endpoint candidates that remain sparse o...,deferred_endpoint_resolution,True,endpoint_crosswalk,ambiguous_candidate,role_ambiguous_no_usable_primary_or_sparse_signal,high,0.927374,52,52,NaN,role_ambiguous_no_usable_primary_or_sparse_signal,NaN,[fill in during cohort blueprint review]
1,deferred_for_later_expansion,endpoint,clinical_patient,days_to_patient_progression_free,deferred_endpoint_candidate,Defer endpoint candidates that remain sparse o...,deferred_endpoint_resolution,True,endpoint_crosswalk,ambiguous_candidate,role_ambiguous_no_usable_primary_or_sparse_signal,high,1.000000,0,0,NaN,role_ambiguous_no_usable_primary_or_sparse_signal,NaN,[fill in during cohort blueprint review]
2,deferred_for_later_expansion,endpoint,clinical_patient,days_to_tumor_progression,deferred_endpoint_candidate,Defer endpoint candidates that remain sparse o...,deferred_endpoint_resolution,True,endpoint_crosswalk,ambiguous_candidate,role_ambiguous_no_usable_primary_or_sparse_signal,high,1.000000,0,0,NaN,role_ambiguous_no_usable_primary_or_sparse_signal,NaN,[fill in during cohort blueprint review]
3,deferred_for_later_expansion,endpoint,clinical_patient,death_days_to,deferred_endpoint_candidate,Defer endpoint candidates that remain sparse o...,deferred_endpoint_resolution,True,endpoint_crosswalk,ambiguous_candidate,role_ambiguous_no_usable_primary_or_sparse_signal,high,0.905196,104,101,NaN,role_ambiguous_no_usable_primary_or_sparse_signal,NaN,[fill in during cohort blueprint review]
4,deferred_for_later_expansion,biospecimen,biospecimen_aliquot,bcr_aliquot_barcode,child_biospecimen_identifier,Keep deeper biospecimen identifiers for later ...,later_child_layer_expansion,True,biospecimen_identifier_crosswalk,primary_link_candidate,role_primary_canonical_level_identifier,low,0.000000,14538,14538,bcr_aliquot_uuid,exact_pair_one_to_one_same_table,1.000000,[fill in during cohort blueprint review]
5,deferred_for_later_expansion,biospecimen,biospecimen_aliquot,bcr_aliquot_uuid,child_biospecimen_identifier,Keep deeper biospecimen identifiers for later ...,later_child_layer_expansion,True,biospecimen_identifier_crosswalk,primary_link_candidate,role_primary_canonical_level_identifier,low,0.000000,14538,14538,bcr_aliquot_barcode,exact_pair_one_to_one_same_table,1.000000,[fill in during cohort blueprint review]
6,deferred_for_later_expansion,biospecimen,biospecimen_aliquot,bcr_patient_uuid,later_biospecimen_side_or_overlap_identifier,Keep overlapping and side-table biospecimen id...,later_side_link_review,True,biospecimen_identifier_crosswalk,overlapping_link_candidate,role_overlapping_repeated_strong_identifier,medium,0.000000,14538,1101,bcr_sample_barcode,same_row_copresence_only,1.000000,[fill in during cohort blueprint review]
7,deferred_for_later_expansion,biospecimen,biospecimen_aliquot,bcr_sample_barcode,later_biospecimen_side_or_overlap_identifier,Keep overlapping and side-table biospecimen id...,later_side_link_review,True,biospecimen_identifier_crosswalk,overlapping_link_candidate,role_overlapping_repeated_strong_identifier,medium,0.000000,14538,2300,bcr_aliquot_barcode,barcode_prefix_same_row,0.999587,[fill in during cohort blueprint review]
8,deferred_for_later_expansion,biospecimen,biospecimen_aliquot,biospecimen_barcode_bottom,ambiguous_biospecimen_identifier,Keep deeper biospecimen identifiers for later ...,manual_identifier_review,True,biospecimen_identifier_crosswalk,ambiguous_candidate,role_ambiguous_generic_or_conflicted_identifier,high,0.000000,14538,14537,bcr_sample_barcode,same_row_copresence_only,1.000000,[fill 

## Save join-path, ambiguity, and summary review tables


In [5]:
join_path_review_df = (
    join_path_df[
        [
            'step_order',
            'left_node',
            'right_node',
            'left_table_name',
            'right_table_name',
            'left_field_name',
            'right_field_name',
            'join_keys_or_evidence',
            'join_strength',
            'link_timing',
            'rationale',
            'ambiguity_flag',
            'notes_placeholder',
        ]
    ]
    .sort_values(['step_order'], ascending=[True])
    .reset_index(drop=True)
)
join_path_review_path = results_root / '51_cohort_blueprint_join_path.tsv'
join_path_review_df.to_csv(join_path_review_path, sep='\t', index=False)

ambiguity_review_df = (
    ambiguity_df[
        [
            'ambiguity_id',
            'ambiguity_category',
            'source_layer',
            'table_name',
            'field_name_or_step',
            'severity',
            'why_unresolved',
            'current_saved_evidence',
            'proposed_blueprint_handling',
            'blocks_final_cohort_construction',
            'notes_placeholder',
        ]
    ]
    .sort_values(['severity', 'ambiguity_id'], ascending=[True, True])
    .reset_index(drop=True)
)
ambiguity_review_path = results_root / '52_cohort_blueprint_ambiguities.tsv'
ambiguity_review_df.to_csv(ambiguity_review_path, sep='\t', index=False)

summary_review_df = summary_df.sort_values(['summary_section', 'summary_metric']).reset_index(drop=True)
summary_review_path = results_root / '53_cohort_blueprint_summary.tsv'
summary_review_df.to_csv(summary_review_path, sep='\t', index=False)

print(f'Saved: {join_path_review_path}')
print(f'Saved: {ambiguity_review_path}')
print(f'Saved: {summary_review_path}')
display(join_path_review_df)
display(ambiguity_review_df)
display(summary_review_df)


Saved: D:\Projects\brcapath-rx\09-trials\01-tcga-only-source-audited\05-results\51_cohort_blueprint_join_path.tsv
Saved: D:\Projects\brcapath-rx\09-trials\01-tcga-only-source-audited\05-results\52_cohort_blueprint_ambiguities.tsv
Saved: D:\Projects\brcapath-rx\09-trials\01-tcga-only-source-audited\05-results\53_cohort_blueprint_summary.tsv


,step_order,left_node,right_node,left_table_name,right_table_name,left_field_name,right_field_name,join_keys_or_evidence,join_strength,link_timing,rationale,ambiguity_flag,notes_placeholder
0,1,patient/case,clinical_patient,NaN,clinical_patient,case_identifier,bcr_patient_barcode / bcr_patient_uuid,clinical_patient carries both audited case ide...,strong_current_evidence,link_now,Use clinical_patient as the central case-level...,False,[fill in during cohort blueprint review]
1,2,patient/case,clinical_drug,clinical_patient,clinical_drug,bcr_patient_barcode / bcr_patient_uuid,bcr_patient_barcode / bcr_patient_uuid,clinical_drug contains repeated patient identi...,workable_but_needs_review,link_later,Treatment rows can be attached later as option...,True,[fill in during cohort blueprint review]
2,3,patient/case,clinical_radiation,clinical_patient,clinical_radiation,bcr_patient_barcode / bcr_patient_uuid,bcr_patient_barcode / bcr_patient_uuid,clinical_radiation contains repeated patient i...,workable_but_needs_review,link_later,Radiation rows should remain optional treatmen...,True,[fill in during cohort blueprint review]
3,4,patient/case,clinical_patient endpoint-like fields,clinical_patient,clinical_patient,bcr_patient_barcode / bcr_patient_uuid,vital_status / last_contact_days_to / tumor_st...,clinical_patient endpoint-like fields already ...,strong_current_evidence,prepare_only,Patient-table endpoint candidates can be carri...,False,[fill in during cohort blueprint review]
4,5,patient/case,clinical_follow_up_v4_0 endpoint-like fields,clinical_patient,clinical_follow_up_v4_0,bcr_patient_barcode / bcr_patient_uuid,bcr_patient_barcode / bcr_patient_uuid,follow-up endpoint-like fields reuse patient i...,workable_but_needs_review,prepare_only,Follow-up endpoint candidates are usable for l...,True,[fill in during cohort blueprint review]
5,6,patient/case,biospecimen_sample,clinical_patient,biospecimen_sample,bcr_patient_uuid / bcr_patient_barcode,bcr_patient_uuid / bcr_sample_barcode / bcr_sa...,biospecimen_sample provides the main specimen ...,workable_but_needs_review,link_now,The blueprint can anchor on biospecimen_sample...,True,[fill in during cohort blueprint review]
6,7,sample,portion,biospecimen_sample,biospecimen_portion,bcr_sample_barcode / bcr_sample_uuid,bcr_sample_barcode / bcr_portion_barcode / bcr...,biospecimen_portion repeats bcr_sample_barcode...,strong_current_evidence,link_later,Sample-to-portion is the strongest saved child...,False,[fill in during cohort blueprint review]
7,8,portion,analyte,biospecimen_portion,biospecimen_analyte,bcr_portion_barcode / bcr_portion_uuid,bcr_analyte_barcode / bcr_analyte_uuid / subpo...,analyte rows retain sample barcode plus subpor...,workable_but_needs_review,link_later,Portion-to-analyte looks workable but still ne...,True,[fill in during cohort blueprint review]
8,9,analyte,slide,biospecimen_analyte,biospecimen_slide,bcr_analyte_barcode / bcr_analyte_uuid,bcr_slide_barcode / bcr_slide_uuid / bcr_sampl...,No direct saved analyte identifier appears in ...,deferred,link_later,An analyte-to-slide step should remain deferre...,True,[fill in during cohort blueprint review]
9,10,analyte,aliquot,biospecimen_analyte,biospecimen_aliquot,bcr_analyte_barcode / bcr_analyte_uuid,bcr_aliquot_barcode / bcr_aliquot_uuid / bcr_s...,No direct saved analyte identifier appears in ...,deferred,link_later,An analyte-to-aliquot step should remain defer...,True,[fill in during cohort blueprint review]


,ambiguity_id,ambiguity_category,source_layer,table_name,field_name_or_step,severity,why_unresolved,current_saved_evidence,proposed_blueprint_handling,blocks_final_cohort_construction,notes_placeholder
0,A01,endpoint_overlap_patient_vs_followup,endpoint,clinical_patient | clinical_follow_up_v4_0,vital_status,high,The same endpoint-like field exists in both pa...,clinical_patient.vital_status has 1097 non-mis...,Carry both rows as endpoint join-preparation f...,True,[fill in during cohort blueprint review]
1,A02,endpoint_overlap_patient_vs_followup,endpoint,clinical_patient | clinical_follow_up_v4_0,last_contact_days_to,high,Patient and follow-up tables both carry last-c...,clinical_patient.last_contact_days_to has 993 ...,Keep both rows optional and endpoint-preparato...,True,[fill in during cohort blueprint review]
2,A03,endpoint_overlap_patient_vs_followup,endpoint,clinical_patient | clinical_follow_up_v4_0,tumor_status,high,Tumor-status signals overlap across patient an...,clinical_patient.tumor_status has 972 non-miss...,Preserve both rows for later endpoint review a...,True,[fill in during cohort blueprint review]
3,A04,endpoint_overlap_patient_vs_followup,endpoint,clinical_patient | clinical_follow_up_v4_0,new_tumor_event_dx_indicator,high,New-tumor-event indicators overlap across pati...,clinical_patient.new_tumor_event_dx_indicator ...,Keep both rows as later endpoint-reconciliatio...,True,[fill in during cohort blueprint review]
4,A05,sparse_endpoint_timing,endpoint,clinical_patient | clinical_follow_up_v4_0,death_days_to,high,Death timing exists in both tables but remains...,clinical_patient.death_days_to has 104 non-mis...,Record death timing as deferred endpoint evide...,True,[fill in during cohort blueprint review]
5,A06,all_missing_progression_timing,endpoint,clinical_patient,days_to_patient_progression_free | days_to_tum...,high,Progression timing fields were surfaced only a...,Both progression timing fields have 0 non-miss...,Keep both fields deferred and exclude them fro...,True,[fill in during cohort blueprint review]
6,A07,clinical_vs_biospecimen_patient_identifiers,clinical | biospecimen,clinical_patient | biospecimen_sample | biospe...,bcr_patient_barcode | bcr_patient_uuid,high,Clinical and biospecimen layers expose both pa...,clinical_patient has 1097 non-missing values f...,Use both identifier forms as evidence-bearing ...,True,[fill in during cohort blueprint review]
7,A08,case_count_mismatch,clinical | source | biospecimen,clinical_patient | source_metadata | biospecim...,case coverage counts,high,"The saved clinical, source-acquisition, and bi...",clinical_patient row count is 1097; clinical s...,Flag cross-layer case counts as a blocking rec...,True,[fill in during cohort blueprint review]
8,A09,treatment_table_multiplicity,clinical,clinical_drug | clinical_radiation,patient-linked treatment rows,medium,Treatment tables are one-to-many and include o...,clinical_drug has 2406 rows over 780 distinct ...,Keep treatment fields optional and proxy-only ...,False,[fill in during cohort blueprint review]
9,A10,missing_direct_child_link,biospecimen,biospecimen_analyte | biospecimen_slide,analyte -> slide,medium,The saved biospecimen crosswalk does not expos...,biospecimen_slide retains sample barcode and s...,Keep analyte-to-slide deferred and use sample-...,False,[fill in during cohort blueprint review]


,blueprint_run_id,summary_section,summary_metric,summary_value,notes
0,20260413T081835Z,ambiguities,ambiguity_count,11,Total unresolved issues recorded by the bluepr...
1,20260413T081835Z,ambiguities,blocking_ambiguity_count,8,Ambiguities currently marked as blocking final...
2,20260413T081835Z,count_signals,biospecimen_patient_uuid_distinct_count,1101,Distinct patient UUID count from the biospecim...
3,20260413T081835Z,count_signals,biospecimen_sample_anchor_count,2302,Sample-anchor row count from the biospecimen c...
4,20260413T081835Z,count_signals,clinical_patient_case_count,1097,clinical_patient row count from the saved clin...
5,20260413T081835Z,count_signals,clinical_source_case_submitter_coverage,1098,Unique clinical source case coverage computed ...
6,20260413T081835Z,design,endpoint_policy,candidate_join_preparation_only,Endpoint rows remain preparatory only; the fin...
7,20260413T081835Z,design,proposed_biospecimen_anchor,sample,biospecimen_sample is the required specimen an...
8,20260413T081835Z,design,proposed_unit_of_analysis,patient/case,The first baseline blueprint keeps the analysi...
9,20260413T081835Z,design,treatment_policy,proxy_only_optional,Treatment rows remain optional proxy evidence ...


## Review reminder

These outputs remain a blueprint and study-design layer only. They do not create a final cohort table, freeze the endpoint, harmonize model-ready variables, parse new raw files, or perform modeling.
